# CSCI E-89 Deep Learning · Assignment 10 · Problem 1
## Natural language → SQL with the OpenAI Responses API

**Student:** Hussein Jawad
**Course:** CSCI E-89 Deep Learning, Harvard Extension School, Fall 2026
**Lecture / lab used:** Lecture 10 and Lab 10 *(LLMs, OpenAI API, Ollama)*, plus *OpenAI Notebook 1* (prompting principles, sampling settings) and *OpenAI Notebook 2* (SQL generation, verification).

**The task.** Use the OpenAI Python API to turn this English sentence into a SQL statement:

> *"Select all customers who are born in either June, July, October, live in Boston, are senior citizens and hence qualify for Medicare benefit"*

| Part | What is asked | Where it is in this notebook |
|---|---|---|
| — | Create an OpenAI account and API key; save the key and its name | [Step 0](#step0) |
| a) | Install the Python OpenAI client library | [Step 1](#step1) |
| b) | Ask OpenAI to build the SQL statement from the English text | [Steps 2–3](#step2) |
| c) | Choose the prompt; vary temperature and other parameters and check accuracy | [Steps 3–5](#step3) |
| d) | Return the result as a JSON response and document it | [Step 6](#step6) |

**Approach in one paragraph.** Following Notebook 2, the database schema goes in the **system** message and the request goes in the **user** message. I first run that baseline prompt as it appears in the lab. Then I build an engineered prompt with the two prompting principles from Notebook 1: delimiters, structured (JSON) output, condition checks, and "work it out first". I ask for the answer through **Structured Outputs** (`text.format = json_schema`), so the response is always valid JSON that matches a schema I define. Notebook 2 says *"Don't trust generated SQL. Verify it."*, so every generated query is run on a small in-memory SQLite database whose correct answer I compute independently. Last, I vary the **prompt, model, temperature, top_p and reasoning effort** and measure accuracy and consistency.

<a id="step0"></a>
## Step 0 · OpenAI account, API key and key name

These steps follow Lab 10 slides 9–10 *(OpenAI with Python; API key and usage)*.

1. Sign up at **https://platform.openai.com/signup** with your email address and phone number.
2. Add a payment method / credits under **Settings → Organization → Billing** (https://platform.openai.com/settings/organization/billing/overview). API usage is billed separately from a ChatGPT subscription. This whole notebook costs only a few cents.
3. Go to **API keys → Create new secret key**. Give it a **name** (I used `e89-hw10-key`) and click *Create*.
4. **Copy the key right away.** OpenAI shows it only once.
5. **Store it safely** in a file named `.env` next to this notebook (never in the notebook itself, and never committed to Git; `.gitignore` already excludes it):

```
OPENAI_API_KEY=sk-...your key...
OPENAI_KEY_NAME=e89-hw10-key
```

6. Check your spending on the **Usage** page.

*Screenshot of the API-keys page (key name visible, secret value hidden):*

<!-- Replace the line below with your screenshot, e.g. ![API key](images/api_key.png) -->
`[insert screenshot images/api_key.png here]`

<a id="step1"></a>
## Step 1 (part a) · Install the Python OpenAI client library

As in the lecture, install the client library `openai` and `python-dotenv` (it reads the `.env` file). `pandas` is used to tabulate the experiment results. Restart the kernel after the first install if Jupyter asks you to.

In [ ]:
%pip install -q --upgrade openai python-dotenv pandas

In [ ]:
import openai, sys
print("Python        :", sys.version.split()[0])
print("openai library:", openai.__version__)

### Load the API key and create the client

This cell is the same as the Setup cell in the lab notebooks. The key is read from `.env` and only a masked version is printed. The model names are kept in two variables so they are easy to change if OpenAI retires a model.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(), override=True)   # override=True: the .env file wins over old environment values

MODEL = "gpt-5.4"              # main model (same as Lab 10)
SMALL_MODEL = "gpt-5.4-mini"   # smaller model, used in the comparison experiments

client = OpenAI()              # reads OPENAI_API_KEY from the environment

key = os.getenv("OPENAI_API_KEY")
assert key, "OPENAI_API_KEY not found. Check your .env file."
print("API key name  :", os.getenv("OPENAI_KEY_NAME", "(set OPENAI_KEY_NAME in .env)"))
print("API key loaded:", key[:3] + "..." + key[-4:])

Quick check that the key works and can use both models. This is a short version of the model-listing cell in Notebook 1. If a model is missing, change `MODEL` / `SMALL_MODEL` above.

In [ ]:
available = {m.id for m in client.models.list().data}
for name in (MODEL, SMALL_MODEL):
    print(f"{name:15s} available: {name in available}")

In [ ]:
import json

def show_json(obj):
    # Pretty-print an API response object as a dictionary (helper from Lab 10 notebooks).
    display(json.loads(obj.model_dump_json()))

<a id="step2"></a>
## Step 2 (part b) · Give the model the database schema

A model can only write correct SQL if it knows the table and column names, so, as in Notebook 2, the schema is part of the system message. The assignment gives no schema, so I defined a realistic `Customers` table. Its `DateOfBirth` column lets us find the birth month and compute the age. There is also an `Orders` table that the query should **not** need; it tests whether the model picks the right table.

Medicare eligibility (the "senior citizen" part) is based on age: **65 or older**. The request says this only indirectly ("are senior citizens and hence qualify for Medicare"), so we will see whether the model works out the age threshold itself.

In [ ]:
SCHEMA = '''
CREATE TABLE Customers (
  CustomerID   INTEGER PRIMARY KEY,
  FirstName    VARCHAR(50),
  LastName     VARCHAR(50),
  DateOfBirth  DATE,          -- stored as text 'YYYY-MM-DD'
  Email        VARCHAR(100),
  Phone        VARCHAR(20),
  Street       VARCHAR(100),
  City         VARCHAR(50),
  State        CHAR(2),       -- two-letter US state code, e.g. 'MA'
  ZipCode      VARCHAR(10)
);

CREATE TABLE Orders (
  OrderID      INTEGER PRIMARY KEY,
  CustomerID   INTEGER REFERENCES Customers(CustomerID),
  OrderDate    DATE,
  TotalAmount  DECIMAL(10, 2)
);
'''.strip()

REQUEST = ("Select all customers who are born in either June, July, October, live in Boston, "
           "are senior citizens and hence qualify for Medicare benefit")
print(SCHEMA)

### 2.1 Baseline: the Notebook 2 prompt, unchanged

This is the same pattern as the lab's *"How to write SQL queries with OpenAI"* cell: the schema in the system message, the request in the user message, `temperature=0` with `reasoning={"effort": "none"}`, and `max_output_tokens=1000`.

In [ ]:
baseline_messages = [
    {"role": "system",
     "content": "Given the following SQL tables, your job is to write queries given a user's request.\n\n" + SCHEMA},
    {"role": "user", "content": REQUEST},
]

baseline_response = client.responses.create(
    model=MODEL,
    input=baseline_messages,
    reasoning={"effort": "none"},   # keeps sampling controls (temperature/top_p) available
    temperature=0,
    max_output_tokens=1000,
)
print("status:", baseline_response.status)
print(baseline_response.output_text)

The baseline usually returns sensible SQL, but as **free text**: the query is wrapped in a Markdown code fence, often with explanation around it, and in a SQL dialect the model chooses (often MySQL or PostgreSQL functions such as `MONTH()`, `EXTRACT()`, `CURDATE()`, `AGE()`). A program cannot reliably use that output. This is what parts (c) and (d) ask us to fix.

<a id="step3"></a>
## Step 3 (parts b, c, d) · Engineered prompt with a JSON (Structured Outputs) response

The engineered prompt applies the prompting principles from **OpenAI Notebook 1**:

| Tactic from Notebook 1 | How it is used here |
|---|---|
| **P1-T1 Delimiters** | The English request is wrapped in `<request> … </request>` and the schema in `<schema> … </schema>`. |
| **P1-T2 Structured output** | The answer must be a JSON object. I enforce this with **Structured Outputs** (`text={"format": {"type": "json_schema", "strict": True, ...}}`), which makes the reply always parse and always match the schema, instead of only asking for JSON in the prompt. |
| **P1-T3 Check conditions** | If the request cannot be answered from the schema, the model must return `"sql": null` and explain why. |
| **P2-T1/T2 Specify steps, work it out first** | The JSON fields are generated in order: the model first lists the **conditions** it extracted (e.g. *age ≥ 65*), then any **assumptions**, and only then writes the **SQL**. |
| **Clear and specific instructions** | Target dialect **SQLite**, read-only `SELECT`, no hard-coded "today", exact age (birthday this year or not), and Boston = Boston, MA. |

I chose SQLite as the target dialect so the generated query can be **executed and verified** inside this notebook (Step 4), the same way Notebook 2 checks its query.

In [ ]:
SYSTEM_PROMPT = f'''
You are an expert SQL developer. Translate the user's request, written in English and
delimited by <request></request>, into ONE SQL statement for the database below.

<schema>
{SCHEMA}
</schema>

Rules:
1. Target dialect: SQLite 3. Use only tables and columns that exist in the schema.
2. Write a single read-only SELECT statement (never INSERT, UPDATE, DELETE, DROP, ...).
3. Dates are stored as text 'YYYY-MM-DD'. Use SQLite date functions (strftime, date).
   Never hard-code today's date; use date('now') so the query stays correct over time.
4. When a condition depends on age, compute the exact age (it matters whether the
   birthday has already happened this year).
5. "Boston" means City = 'Boston' and State = 'MA'.
6. "Select all customers" means return every column of the matching customers.
7. Work it out first: list each condition you extracted from the request (with any
   domain knowledge you used) BEFORE you write the query.
8. If the request cannot be answered with this schema, set "sql" to null and explain why.

Respond only with JSON that matches the provided JSON schema.
'''.strip()

def user_message(request):
    return f"<request>{request}</request>"

print(SYSTEM_PROMPT)

The JSON schema that the model's answer must follow (Structured Outputs, `strict: true`):

In [ ]:
SQL_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "natural_language_request": {"type": "string",
            "description": "The user's request, copied verbatim."},
        "conditions": {"type": "array", "items": {"type": "string"},
            "description": "Each filter condition extracted from the request, in plain English."},
        "assumptions": {"type": "array", "items": {"type": "string"},
            "description": "Domain knowledge or interpretation choices used (e.g. Medicare age)."},
        "dialect": {"type": "string", "description": "SQL dialect of the query."},
        "tables_used": {"type": "array", "items": {"type": "string"}},
        "sql": {"type": ["string", "null"],
            "description": "The SQL statement, or null if the request cannot be answered."},
        "explanation": {"type": "string",
            "description": "Short explanation of how the query implements each condition."},
    },
    "required": ["natural_language_request", "conditions", "assumptions", "dialect",
                 "tables_used", "sql", "explanation"],
    "additionalProperties": False,
}

TEXT_FORMAT = {"format": {"type": "json_schema", "name": "nl_to_sql",
                          "schema": SQL_JSON_SCHEMA, "strict": True}}

### 3.1 One helper used for every call

`generate_sql()` sends one request with the chosen prompt version, model and sampling settings, and returns the full response object. It follows the Notebook 1 rules for sampling and reasoning:

* `reasoning.effort = "none"` lets us set `temperature` / `top_p`.
* If `effort` is anything else (`low`, `medium`, ...), we **must not** send `temperature` or `top_p`. Sending them causes an API error.

In [ ]:
def generate_sql(prompt_version="engineered", model=MODEL, temperature=None, top_p=None,
                 effort="none", max_output_tokens=1000):
    if prompt_version == "baseline":
        kwargs = {"input": baseline_messages}
    else:
        kwargs = {"input": [{"role": "system", "content": SYSTEM_PROMPT},
                            {"role": "user",   "content": user_message(REQUEST)}],
                  "text": TEXT_FORMAT}
    kwargs["reasoning"] = {"effort": effort}
    if effort == "none":                       # sampling controls only allowed without reasoning
        if temperature is not None: kwargs["temperature"] = temperature
        if top_p is not None:       kwargs["top_p"] = top_p
    else:                                      # reasoning tokens count toward the output budget
        max_output_tokens = max(max_output_tokens, 4000)
    return client.responses.create(model=model, max_output_tokens=max_output_tokens, **kwargs)

### 3.2 Engineered prompt, `temperature = 0`

In [ ]:
engineered_response = generate_sql("engineered", MODEL, temperature=0)

print("status            :", engineered_response.status)
print("incomplete_details:", engineered_response.incomplete_details)
print("usage             :", engineered_response.usage.input_tokens, "in /",
      engineered_response.usage.output_tokens, "out tokens")

result = json.loads(engineered_response.output_text)   # always valid JSON thanks to Structured Outputs
print(json.dumps(result, indent=2))
print("\nGenerated SQL:\n" + result["sql"])

Check the `conditions` and `assumptions` fields: the model should have worked out on its own that "qualify for Medicare" means **age ≥ 65**, and that the three months are 6, 7 and 10.

<a id="step4"></a>
## Step 4 (part c) · Verify the generated SQL against a known answer

As Notebook 2 says: *"A query that looks right can still be wrong, and a wrong query often fails silently."* So I measure accuracy objectively instead of reading the SQL by eye:

1. Build a small in-memory SQLite database with the same tables. The customers are chosen to test **each condition separately**: right month but wrong city, right city but too young, the boundary age of exactly 65, a missing birth date, and so on.
2. Compute the **correct answer in Python**, independently of SQL (exact age, month in {6, 7, 10}, City = Boston, State = MA).
3. Run the generated query **read-only** (`PRAGMA query_only = ON`, an extra safeguard that Notebook 2's exercises recommend) and compare the returned `CustomerID`s with the expected set.

Birth dates are created relative to *today*, so the test stays correct whenever the notebook is run.

In [ ]:
import re, sqlite3
from datetime import date, datetime, timezone

TODAY = datetime.now(timezone.utc).date()     # SQLite's date('now') is UTC, so use UTC here too

def years_ago(n, month, day):
    return date(TODAY.year - n, month, day).isoformat()

# (CustomerID, First, Last, DateOfBirth, City, State, why it is in the test)
TEST_CUSTOMERS = [
    (1,  "Mary",    "June",     years_ago(72, 6, 15),  "Boston",    "MA", "June, Boston, 72 -> match"),
    (2,  "John",    "July",     years_ago(80, 7, 4),   "Boston",    "MA", "July, Boston, 80 -> match"),
    (3,  "Ann",     "October",  years_ago(66, 10, 31), "Boston",    "MA", "October, Boston, 66 -> match"),
    (4,  "Eve",     "Boundary", years_ago(65, 10, 1),  "Boston",    "MA", "exactly 65 if birthday passed (tests >= vs >)"),
    (5,  "Paul",    "Turning",  years_ago(65, 7, 31),  "Boston",    "MA", "65 this year; depends on today's date"),
    (6,  "Bob",     "May",      years_ago(75, 5, 10),  "Boston",    "MA", "wrong month (May)"),
    (7,  "Grace",   "December", years_ago(90, 12, 1),  "Boston",    "MA", "wrong month (December)"),
    (8,  "Sue",     "January",  years_ago(70, 1, 6),   "Boston",    "MA", "wrong month (January)"),
    (9,  "Carl",    "Cambridge",years_ago(70, 6, 1),   "Cambridge", "MA", "wrong city"),
    (10, "Nina",    "Worcester",years_ago(68, 10, 12), "Worcester", "MA", "wrong city"),
    (11, "Dana",    "Young",    years_ago(40, 6, 20),  "Boston",    "MA", "too young (40)"),
    (12, "Frank",   "Almost",   years_ago(64, 7, 15),  "Boston",    "MA", "64 or 63 -> too young"),
    (13, "Omar",    "Unknown",  None,                  "Boston",    "MA", "missing birth date"),
    (14, "Lena",    "Senior",   years_ago(85, 6, 30),  "Boston",    "MA", "June, Boston, 85 -> match"),
]

def build_test_db():
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    con.executemany(
        "INSERT INTO Customers (CustomerID, FirstName, LastName, DateOfBirth, Email, Phone, Street, City, State, ZipCode) "
        "VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        [(cid, fn, ln, dob, f"{fn.lower()}@example.com", "617-555-01%02d" % cid,
          f"{cid} Main St", city, st, "02108" if city == "Boston" else "02139")
         for cid, fn, ln, dob, city, st, _ in TEST_CUSTOMERS])
    con.executemany("INSERT INTO Orders VALUES (?, ?, ?, ?)",
                    [(1, 1, "2026-01-05", 120.50), (2, 9, "2026-02-11", 33.10)])
    con.execute("PRAGMA query_only = ON")          # the generated SQL cannot modify anything
    return con

def exact_age(dob_iso):
    d = date.fromisoformat(dob_iso)
    return TODAY.year - d.year - ((TODAY.month, TODAY.day) < (d.month, d.day))

# Ground truth computed in Python, independently of any SQL
EXPECTED_IDS = {cid for cid, _, _, dob, city, st, _ in TEST_CUSTOMERS
                if dob and date.fromisoformat(dob).month in (6, 7, 10)
                and city == "Boston" and st == "MA" and exact_age(dob) >= 65}

import pandas as pd
pd.set_option("display.max_colwidth", 80)
test_df = pd.DataFrame(TEST_CUSTOMERS, columns=["CustomerID", "First", "Last", "DateOfBirth", "City", "State", "Purpose"])
test_df["Age"] = test_df["DateOfBirth"].map(lambda d: exact_age(d) if isinstance(d, str) else None)
test_df["Expected"] = test_df["CustomerID"].isin(EXPECTED_IDS)
print("Today (UTC):", TODAY, "   Expected CustomerIDs:", sorted(EXPECTED_IDS))
test_df

`check_sql()` grades one query. It extracts the SQL (from JSON, or from a ```` ```sql ```` fence for the free-text baseline), refuses anything that is not a `SELECT`, runs it, and compares the result with `EXPECTED_IDS`.

In [ ]:
def extract_sql(text):
    # SQL from a free-text reply: the content of the first ```sql fence, or the whole text.
    m = re.search(r"```(?:sql|sqlite)?\s*(.*?)```", text, re.S | re.I)
    return (m.group(1) if m else text).strip()

def check_sql(sql):
    # Run `sql` on the test DB and compare with EXPECTED_IDS. Returns a dict of results.
    out = {"runs_on_sqlite": False, "correct": False, "returned_ids": None, "error": None}
    if not sql:
        out["error"] = "no SQL returned"; return out
    if not re.match(r"^\s*(WITH|SELECT)\b", sql, re.I):
        out["error"] = "not a read-only SELECT statement - refused"; return out
    try:
        cur = build_test_db().execute(sql)
        cols = [c[0].lower() for c in cur.description]
        rows = cur.fetchall()
        out["runs_on_sqlite"] = True
        if "customerid" not in cols:
            out["error"] = f"no CustomerID column in result {cols}"; return out
        ids = {r[cols.index("customerid")] for r in rows}
        out["returned_ids"] = sorted(ids)
        out["correct"] = (ids == EXPECTED_IDS)
    except Exception as e:                        # e.g. MySQL-only functions such as MONTH()/CURDATE()
        out["error"] = f"{type(e).__name__}: {e}"
    return out

# Sanity check of the grader with my own hand-written reference query
REFERENCE_SQL = '''
SELECT *
FROM Customers
WHERE CAST(strftime('%m', DateOfBirth) AS INTEGER) IN (6, 7, 10)
  AND City = 'Boston' AND State = 'MA'
  AND DateOfBirth <= date('now', '-65 years');
'''
print("Reference query :", check_sql(REFERENCE_SQL))
print("Wrong query (>) :", check_sql(REFERENCE_SQL.replace("<=", "<").replace("-65", "-66")))

The grader accepts the reference query and rejects a query with an off-by-one age error. Now grade the two model answers from Steps 2 and 3:

In [ ]:
baseline_sql = extract_sql(baseline_response.output_text)
print("BASELINE (Notebook 2 prompt):\n", baseline_sql, "\n->", check_sql(baseline_sql), "\n")

engineered_sql = result["sql"]
print("ENGINEERED (JSON, SQLite):\n", engineered_sql, "\n->", check_sql(engineered_sql))

In [ ]:
# Show the rows the engineered query returns from the test database
pd.read_sql_query(engineered_sql, build_test_db())

<a id="step5"></a>
## Step 5 (part c) · Experiments: prompt, model, temperature, top_p and reasoning effort

Notebook 1 *("Adjust your settings")* explains the settings:

* **temperature** (0–2): how random the token choice is. `0` gives nearly deterministic output, which suits code and data extraction. A high value gives more variety and a higher risk of odd output.
* **top_p** (0–1): sample only from the smallest set of tokens whose probabilities add up to *p*. OpenAI recommends adjusting temperature **or** top_p, not both, so the top_p runs leave temperature at its default.
* **reasoning effort**: lets the model "think" before answering. With any effort other than `none`, temperature and top_p cannot be sent.
* **model**: the full `gpt-5.4` against the smaller `gpt-5.4-mini`.

Each configuration runs `N_RUNS` times. For every run I record whether the output was valid JSON, whether the SQL runs on SQLite, whether it returns exactly the right customers, the token usage and the latency. Repeated runs show **consistency**: how many different SQL strings one setting produces. The grid makes about 33 small calls (a few cents).

In [ ]:
import time

N_RUNS = 3
CONFIGS = [
    # (label,                         prompt,        model,       settings)
    ("baseline  | T=0",               "baseline",    MODEL,       {"temperature": 0}),
    ("baseline  | T=1.5",             "baseline",    MODEL,       {"temperature": 1.5}),
    ("engineered| T=0",               "engineered",  MODEL,       {"temperature": 0}),
    ("engineered| T=0.7",             "engineered",  MODEL,       {"temperature": 0.7}),
    ("engineered| T=1.5",             "engineered",  MODEL,       {"temperature": 1.5}),
    ("engineered| T=2.0",             "engineered",  MODEL,       {"temperature": 2.0}),
    ("engineered| top_p=0.1",         "engineered",  MODEL,       {"top_p": 0.1}),
    ("engineered| mini T=0",          "engineered",  SMALL_MODEL, {"temperature": 0}),
    ("engineered| mini T=1.5",        "engineered",  SMALL_MODEL, {"temperature": 1.5}),
    ("engineered| effort=low",        "engineered",  MODEL,       {"effort": "low"}),
    ("engineered| effort=medium",     "engineered",  MODEL,       {"effort": "medium"}),
]

def run_one(prompt_version, model, settings):
    row = {"valid_json": None, "sql": None}
    t0 = time.perf_counter()
    try:
        r = generate_sql(prompt_version, model, **settings)
    except Exception as e:                      # e.g. a parameter the model does not support
        return {**row, "status": "api_error", "error": f"{type(e).__name__}: {e}"[:200],
                "runs_on_sqlite": False, "correct": False}
    row["latency_s"] = round(time.perf_counter() - t0, 2)
    row["status"] = r.status
    row["total_tokens"] = r.usage.total_tokens
    if prompt_version == "baseline":
        row["sql"] = extract_sql(r.output_text)
    else:
        try:
            row["sql"] = json.loads(r.output_text)["sql"]; row["valid_json"] = True
        except Exception:
            row["valid_json"] = False
    return {**row, **check_sql(row["sql"])}

records = []
for label, prompt_version, model, settings in CONFIGS:
    for i in range(N_RUNS):
        rec = run_one(prompt_version, model, settings)
        records.append({"config": label, "model": model, "run": i + 1, **rec})
    print(f"done: {label}")

runs = pd.DataFrame(records)

### 5.1 Results per configuration

* **accuracy**: share of runs whose query returned exactly the expected customers.
* **runs_on_sqlite**: share of runs whose query executed at all.
* **distinct_sql**: number of different SQL strings across the `N_RUNS` runs (1 means fully consistent).

In [ ]:
summary = (runs.groupby("config", sort=False)
               .agg(model=("model", "first"),
                    accuracy=("correct", "mean"),
                    runs_on_sqlite=("runs_on_sqlite", "mean"),
                    valid_json=("valid_json", lambda s: s.dropna().mean() if s.notna().any() else None),
                    distinct_sql=("sql", lambda s: s.dropna().str.strip().nunique()),
                    avg_tokens=("total_tokens", "mean"),
                    avg_latency_s=("latency_s", "mean")))
summary.style.format({"accuracy": "{:.0%}", "runs_on_sqlite": "{:.0%}", "valid_json": "{:.0%}",
                      "avg_tokens": "{:.0f}", "avg_latency_s": "{:.2f}"}, na_rep="n/a")

### 5.2 Failures and variety

Every run that failed, with the reason. Afterwards, the different queries produced at the highest temperature, to show how much the SQL text changes even when the logic stays the same.

In [ ]:
failed = runs[~runs["correct"]][["config", "run", "status", "error", "returned_ids", "sql"]]
print(f"{len(failed)} of {len(runs)} runs were not fully correct")
failed

In [ ]:
for label in ["engineered| T=1.5", "engineered| T=2.0"]:
    variants = runs.loc[runs["config"] == label, "sql"].dropna().str.strip().unique()
    print(f"===== {label}: {len(variants)} distinct quer{'y' if len(variants) == 1 else 'ies'} =====")
    for v in variants:
        print(v, "\n---")

### 5.3 What the experiments show

The table above comes from the live run. These are the points to compare it with (based on Lecture 10 / Notebook 1):

1. **Prompt matters more than temperature.** The baseline prompt does not name a dialect, so it often writes MySQL/PostgreSQL date functions (`MONTH()`, `CURDATE()`, `EXTRACT`, `AGE`). That SQL may be logically correct but does not run on the target database, and it comes back as free text, not JSON. Naming the dialect, the date format and the business rules in the system message fixes both problems.
2. **Structured Outputs always produce valid JSON.** With `json_schema` and `strict: true`, `valid_json` should be 100% for every engineered run, at every temperature. Valid JSON does not mean the SQL is correct, so Step 4's check is still needed.
3. **Temperature 0 means consistent, not necessarily correct** (Notebook 1). Low temperature gives the same query almost every run (`distinct_sql` ≈ 1). Higher temperature increases variety, mostly in formatting, aliases or equivalent age formulas. At `T = 2.0` the output can become unreliable or the response can stop early.
4. **top_p = 0.1** narrows the token pool and behaves much like a low temperature.
5. **Reasoning effort** (`low` / `medium`) uses more tokens and takes longer. For a short, well-specified task like this it usually gives no accuracy gain, which matches Notebook 1's rule of thumb: *"move down for simple, well-defined tasks"*.
6. **Smaller model**: `gpt-5.4-mini` is cheaper and faster. Check in the table whether it handles the boundary cases (age exactly 65, `>=` vs `>`) as well as `gpt-5.4`.

**Chosen setting for the final answer:** engineered prompt + Structured Outputs, `gpt-5.4`, `reasoning.effort = "none"`, `temperature = 0`. This is the most accurate and most reproducible combination at the lowest cost.

<a id="step6"></a>
## Step 6 (part d) · Final answer as a JSON response, documented

The final call uses the chosen setting. The submission documents it in three ways:

1. the **complete API response** as JSON (`show_json`, as in the lab), where `output[0].content[0].text` holds the model's JSON answer;
2. the **parsed JSON answer** (the SQL plus the conditions, assumptions and explanation);
3. both saved to files in `results/` and committed with the notebook.

In [ ]:
final_response = generate_sql("engineered", MODEL, temperature=0)
show_json(final_response)       # 1. the full Responses API object, as JSON

In [ ]:
final_answer = json.loads(final_response.output_text)          # 2. the model's JSON answer
final_check = check_sql(final_answer["sql"])

documented = {
    "student": "Hussein Jawad",
    "assignment": "CSCI E-89 Assignment 10, Problem 1",
    "request": REQUEST,
    "model": final_response.model,
    "settings": {"reasoning_effort": "none", "temperature": 0, "max_output_tokens": 1000,
                 "text_format": "json_schema (strict)"},
    "response_id": final_response.id,
    "status": final_response.status,
    "usage": {"input_tokens": final_response.usage.input_tokens,
              "output_tokens": final_response.usage.output_tokens,
              "total_tokens": final_response.usage.total_tokens},
    "answer": final_answer,
    "verification": {"test_date_utc": TODAY.isoformat(),
                     "expected_customer_ids": sorted(EXPECTED_IDS), **final_check},
}
print(json.dumps(documented, indent=2))

In [ ]:
# 3. save the documented JSON response and the full raw API response next to the notebook
os.makedirs("results", exist_ok=True)
with open("results/problem1_final_response.json", "w") as f:
    json.dump(documented, f, indent=2)
with open("results/problem1_full_api_response.json", "w") as f:
    f.write(final_response.model_dump_json(indent=2))
runs.to_csv("results/problem1_experiments.csv", index=False)
print("Saved: results/problem1_final_response.json, results/problem1_full_api_response.json, "
      "results/problem1_experiments.csv")
print("\nFINAL SQL:\n" + final_answer["sql"])

### Summary

* **(a)** Installed `openai` (plus `python-dotenv`). The API key is stored in `.env` under the name shown in Step 1 and is never printed in full or committed.
* **(b)** Used the Responses API to turn the English request into SQL, first with the Lab 10 / Notebook 2 baseline prompt and then with an engineered prompt.
* **(c)** Built the prompt with the Notebook 1 principles. Measured accuracy objectively with a test database and Python-computed ground truth. Compared prompt versions, two models, temperature 0 → 2, top_p and reasoning effort (Step 5 table).
* **(d)** The result is a JSON response enforced by Structured Outputs. It is documented in full above (raw API JSON plus the parsed answer) and saved in `results/problem1_final_response.json`.

**A caution from Notebook 2:** treat model-generated SQL as untrusted code. Review it, test it on data with a known answer, and run it with a read-only database user. Never run it directly on production.